In [1]:
# ============================================================
# PARÁMETROS DEL NOTEBOOK
# ============================================================

LAKEHOUSE = "lh_tinkle_financiero"
ESQUEMA = "200"

# Vistas validadas
VISTA_BALANCE = "vw_Export_Grupo"
VISTA_SALDOS = "vw_Export_Saldos_4_7"

# Carpeta base de salida dentro de Files del Lakehouse predeterminado.
# Se creará una subcarpeta por periodo, por ejemplo 2026.03.
CARPETA_SALIDA = "Files/Exportaciones/Balance"

# Si existe un fichero con el mismo nombre, se sustituye.
SOBRESCRIBIR = True

# Incluye fecha y hora de ejecución en el nombre.
INCLUIR_TIMESTAMP = True

# El Balance de Magnitude no necesita líneas de importe 0.
EXCLUIR_IMPORTES_CERO = False

# D_RU -> prefijo histórico del fichero.
NOMBRES_SOCIEDAD = {
    "ESP066": "TINKLE_COMM",
    "ESP067": "TINKLE_EU",
    "ESP068": "TINKLE_JAZZ",
    "ESP069": "TINKLE_ONION",
    "ESP070": "TINKLE_CONS",
}

print("Parámetros cargados.")


StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 3, Finished, Available, Finished, False)

Parámetros cargados.


In [2]:
# ============================================================
# IMPORTACIONES
# ============================================================

from datetime import datetime
import re
import uuid

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


# Compatibilidad entre versiones de NotebookUtils.
try:
    fs = notebookutils.fs
except NameError:
    from notebookutils import mssparkutils
    fs = mssparkutils.fs


print("Importaciones completadas correctamente.")

StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 4, Finished, Available, Finished, False)

Importaciones completadas correctamente.


In [3]:
# ============================================================
# ESTRUCTURA ESPERADA DEL TXT
# ============================================================

COLUMNAS_TXT = [
    "D_CA",
    "D_DP",
    "D_DEVTR",
    "D_SAJOB",
    "D_DIM2",
    "D_DIM3",
    "D_ACTY",
    "D_CLIENT",
    "D_PE",
    "D_RU",
    "D_ORU",
    "D_AC",
    "D_FL",
    "D_AU",
    "D_T1",
    "D_T2",
    "D_CU",
    "D_TO",
    "D_GO",
    "D_LE",
    "D_NU",
    "D_DIM4",
    "D_DEST",
    "D_AREA",
    "D_TFTR",
    "P_AMOUNT",
    "P_COMMENT",
]

COLUMNAS_OBLIGATORIAS = [
    "D_CA",
    "D_DP",
    "D_PE",
    "D_RU",
    "D_ORU",
    "D_AC",
    "D_FL",
    "D_AU",
    "D_CU",
    "D_TO",
    "D_GO",
    "P_AMOUNT",
]

print(f"Número de columnas esperadas: {len(COLUMNAS_TXT)}")

StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 5, Finished, Available, Finished, False)

Número de columnas esperadas: 27


In [4]:
# ============================================================
# LECTURA DE LAS DOS VISTAS DEL SQL ANALYTICS ENDPOINT
# ============================================================

NOMBRE_COMPLETO_BALANCE = f"{LAKEHOUSE}.{ESQUEMA}.{VISTA_BALANCE}"
NOMBRE_COMPLETO_SALDOS = f"{LAKEHOUSE}.{ESQUEMA}.{VISTA_SALDOS}"

print(f"Leyendo Balance: {NOMBRE_COMPLETO_BALANCE}")
df_origen = spark.read.synapsesql(NOMBRE_COMPLETO_BALANCE)
print(f"Filas Balance: {df_origen.count():,}")

print(f"Leyendo Saldos 4_7: {NOMBRE_COMPLETO_SALDOS}")
df_saldos_origen = spark.read.synapsesql(NOMBRE_COMPLETO_SALDOS)
print(f"Filas Saldos 4_7: {df_saldos_origen.count():,}")

print("Esquema Balance:")
df_origen.printSchema()

print("Esquema Saldos 4_7:")
df_saldos_origen.printSchema()


StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 6, Finished, Available, Finished, False)

Leyendo Balance: lh_tinkle_financiero.200.vw_Export_Grupo
Filas Balance: 224
Leyendo Saldos 4_7: lh_tinkle_financiero.200.vw_Export_Saldos_4_7
Filas Saldos 4_7: 641
Esquema Balance:
root
 |-- D_NU: string (nullable = true)
 |-- D_TFTR: string (nullable = true)
 |-- D_T2: string (nullable = true)
 |-- D_DIM3: string (nullable = true)
 |-- D_ACTY: string (nullable = true)
 |-- D_GO: string (nullable = true)
 |-- D_TO: string (nullable = true)
 |-- D_PE: string (nullable = true)
 |-- D_DEVTR: string (nullable = true)
 |-- D_CA: string (nullable = true)
 |-- D_FL: string (nullable = true)
 |-- D_AREA: string (nullable = true)
 |-- D_DIM2: string (nullable = true)
 |-- D_DEST: string (nullable = true)
 |-- P_COMMENT: string (nullable = true)
 |-- P_AMOUNT: long (nullable = true)
 |-- D_RU: string (nullable = true)
 |-- D_T1: string (nullable = true)
 |-- D_CU: string (nullable = true)
 |-- D_AU: string (nullable = true)
 |-- D_ORU: string (nullable = true)
 |-- D_CLIENT: string (nullable = 

In [5]:
# ============================================================
# PREPARACIÓN Y NORMALIZACIÓN
# ============================================================

columnas_faltantes = [
    columna
    for columna in COLUMNAS_TXT
    if columna not in df_origen.columns
]

if columnas_faltantes:
    raise ValueError(
        "La vista no contiene todas las columnas requeridas. "
        f"Faltan: {columnas_faltantes}"
    )


# Seleccionar exactamente las columnas y en el orden requerido.
df = df_origen.select(*COLUMNAS_TXT)


# Sustituir null por cadenas vacías en campos de texto.
columnas_texto = [
    columna
    for columna in COLUMNAS_TXT
    if columna != "P_AMOUNT"
]

for columna in columnas_texto:
    df = df.withColumn(
        columna,
        F.coalesce(
            F.col(columna).cast("string"),
            F.lit("")
        )
    )


# Normalizar importe.
#
# Se redondea a 0 decimales porque el fichero de ejemplo no contiene
# parte decimal. Si Grupo requiere céntimos, cambia cast("long") por
# cast(DecimalType(18, 2)).
df = df.withColumn(
    "P_AMOUNT",
    F.round(F.col("P_AMOUNT"), 0).cast("long")
)


if EXCLUIR_IMPORTES_CERO:
    df = df.filter(F.col("P_AMOUNT") != 0)


# Orden estable dentro de cada fichero.
df = df.orderBy(
    F.col("D_RU"),
    F.col("D_AC")
)


numero_filas = df.count()

if numero_filas == 0:
    raise ValueError(
        "La vista no devuelve registros. No se generará ningún TXT."
    )

print(f"Filas preparadas: {numero_filas:,}")

StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 7, Finished, Available, Finished, False)

Filas preparadas: 224


In [6]:
# ============================================================
# VALIDACIONES PREVIAS
# ============================================================

def condicion_vacio(nombre_columna: str):
    """Devuelve una condición Spark para detectar nulos o vacíos."""
    if nombre_columna == "P_AMOUNT":
        return F.col(nombre_columna).isNull()

    return (
        F.col(nombre_columna).isNull()
        | (F.trim(F.col(nombre_columna).cast("string")) == "")
    )


# ------------------------------------------------------------
# 1. Campos obligatorios
# ------------------------------------------------------------

condicion_campos_vacios = None

for columna in COLUMNAS_OBLIGATORIAS:
    condicion_actual = condicion_vacio(columna)

    condicion_campos_vacios = (
        condicion_actual
        if condicion_campos_vacios is None
        else condicion_campos_vacios | condicion_actual
    )


errores_obligatorios = df.filter(condicion_campos_vacios)

if errores_obligatorios.limit(1).count() > 0:
    print("Ejemplos de registros con campos obligatorios vacíos:")
    errores_obligatorios.show(20, truncate=False)

    raise ValueError(
        "Existen registros con campos obligatorios vacíos."
    )


# ------------------------------------------------------------
# 2. Formato del periodo YYYY.MM
# ------------------------------------------------------------

periodos_invalidos = df.filter(
    ~F.col("D_DP").rlike(r"^\d{4}\.\d{2}$")
    | ~F.col("D_PE").rlike(r"^\d{4}\.\d{2}$")
)

if periodos_invalidos.limit(1).count() > 0:
    print("Registros con periodos incorrectos:")
    periodos_invalidos.select(
        "D_RU",
        "D_AC",
        "D_DP",
        "D_PE"
    ).show(20, truncate=False)

    raise ValueError(
        "D_DP o D_PE no tienen formato YYYY.MM."
    )


# ------------------------------------------------------------
# 3. D_DP y D_PE deben coincidir
# ------------------------------------------------------------

periodos_distintos = df.filter(
    F.col("D_DP") != F.col("D_PE")
)

if periodos_distintos.limit(1).count() > 0:
    periodos_distintos.select(
        "D_RU",
        "D_AC",
        "D_DP",
        "D_PE"
    ).show(20, truncate=False)

    raise ValueError(
        "Existen filas donde D_DP y D_PE son diferentes."
    )


# ------------------------------------------------------------
# 4. D_RU y D_ORU deben coincidir
# ------------------------------------------------------------

sociedades_distintas = df.filter(
    F.col("D_RU") != F.col("D_ORU")
)

if sociedades_distintas.limit(1).count() > 0:
    sociedades_distintas.select(
        "D_RU",
        "D_ORU",
        "D_AC"
    ).show(20, truncate=False)

    raise ValueError(
        "Existen filas donde D_RU y D_ORU son diferentes."
    )


# ------------------------------------------------------------
# 5. Duplicados por sociedad, periodo y cuenta
# ------------------------------------------------------------

duplicados = (
    df.groupBy(
        "D_DP",
        "D_RU",
        "D_AC"
    )
    .count()
    .filter(F.col("count") > 1)
)

if duplicados.limit(1).count() > 0:
    print("Cuentas duplicadas:")
    duplicados.show(50, truncate=False)

    raise ValueError(
        "Existen cuentas duplicadas para la misma sociedad y periodo."
    )


# ------------------------------------------------------------
# 6. Una sociedad no debe mezclar periodos
# ------------------------------------------------------------

periodos_por_sociedad = (
    df.groupBy("D_RU")
    .agg(
        F.countDistinct("D_DP").alias("NumeroPeriodos")
    )
    .filter(F.col("NumeroPeriodos") > 1)
)

if periodos_por_sociedad.limit(1).count() > 0:
    periodos_por_sociedad.show(truncate=False)

    raise ValueError(
        "Una misma sociedad contiene más de un periodo en la vista."
    )


print("Todas las validaciones se han superado.")

StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 8, Finished, Available, Finished, False)

Todas las validaciones se han superado.


In [7]:
# ============================================================
# PREPARACIÓN Y VALIDACIÓN DE SALDOS 4_7
# ============================================================

COLUMNAS_SALDOS = [
    "Sociedad",
    "CodigoFichero",
    "PeriodoTexto",
    "Periodo",
    "Cuenta",
    "Descripcion",
    "Debe",
    "Haber",
    "Saldo",
]

faltantes_saldos = [
    c for c in COLUMNAS_SALDOS
    if c not in df_saldos_origen.columns
]

if faltantes_saldos:
    raise ValueError(
        "La vista de Saldos 4_7 no contiene todas las columnas requeridas. "
        f"Faltan: {faltantes_saldos}"
    )

# Normalización básica.
df_saldos = (
    df_saldos_origen
    .select(*COLUMNAS_SALDOS)
    .withColumn("Sociedad", F.coalesce(F.col("Sociedad").cast("string"), F.lit("")))
    .withColumn("CodigoFichero", F.coalesce(F.col("CodigoFichero").cast("string"), F.lit("")))
    .withColumn("PeriodoTexto", F.coalesce(F.col("PeriodoTexto").cast("string"), F.lit("")))
    .withColumn("Cuenta", F.coalesce(F.col("Cuenta").cast("string"), F.lit("")))
    .withColumn("Descripcion", F.coalesce(F.col("Descripcion").cast("string"), F.lit("")))
    .withColumn("PeriodoCodigo", F.date_format(F.col("Periodo"), "yyyy.MM"))
    .orderBy("CodigoFichero", "Cuenta")
)

# Validaciones mínimas.
if df_saldos.limit(1).count() == 0:
    raise ValueError("La vista Saldos 4_7 no devuelve registros.")

invalidos_saldos = df_saldos.filter(
    (F.trim(F.col("Sociedad")) == "")
    | (F.trim(F.col("CodigoFichero")) == "")
    | (F.trim(F.col("PeriodoTexto")) == "")
    | F.col("Periodo").isNull()
    | (F.trim(F.col("Cuenta")) == "")
    | F.col("Saldo").isNull()
)

if invalidos_saldos.limit(1).count() > 0:
    print("Ejemplos de filas inválidas en Saldos 4_7:")
    invalidos_saldos.show(20, truncate=False)
    raise ValueError("Existen filas inválidas en Saldos 4_7.")

# Cada sociedad debe tener un único periodo.
periodos_saldos = (
    df_saldos.groupBy("CodigoFichero")
    .agg(F.countDistinct("PeriodoCodigo").alias("NumeroPeriodos"))
    .filter(F.col("NumeroPeriodos") > 1)
)

if periodos_saldos.limit(1).count() > 0:
    periodos_saldos.show(truncate=False)
    raise ValueError("Una sociedad contiene más de un periodo en Saldos 4_7.")

print(f"Filas Saldos 4_7 preparadas: {df_saldos.count():,}")


StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 9, Finished, Available, Finished, False)

Filas Saldos 4_7 preparadas: 641


In [8]:
# ============================================================
# FUNCIONES DE GENERACIÓN
# ============================================================

# Timestamp único para los 10 ficheros de la misma ejecución.
TIMESTAMP_EJECUCION = datetime.now().strftime("%Y%m%d_%H%M%S")

MESES = {
    1: "JAN", 2: "FEB", 3: "MAR", 4: "APR",
    5: "MAY", 6: "JUN", 7: "JUL", 8: "AUG",
    9: "SEP", 10: "OCT", 11: "NOV", 12: "DEC",
}

MESES_ES = {
    1: "ENE", 2: "FEB", 3: "MAR", 4: "ABR",
    5: "MAY", 6: "JUN", 7: "JUL", 8: "AGO",
    9: "SEP", 10: "OCT", 11: "NOV", 12: "DIC",
}


def limpiar_nombre(valor: str) -> str:
    valor = str(valor).strip()
    valor = re.sub(r"[^A-Za-z0-9_-]+", "_", valor)
    valor = re.sub(r"_+", "_", valor)
    return valor.strip("_")


def convertir_periodo_a_nombre(periodo: str) -> str:
    """Convierte 2026.03 en MAR-26."""
    partes = periodo.split(".")
    if len(partes) != 2:
        raise ValueError(f"Periodo no válido: {periodo}")

    anio = int(partes[0])
    mes = int(partes[1])

    if mes not in MESES:
        raise ValueError(f"Mes no válido en el periodo: {periodo}")

    return f"{MESES[mes]}-{str(anio)[-2:]}"


def obtener_nombre_sociedad(codigo_ru: str) -> str:
    return limpiar_nombre(NOMBRES_SOCIEDAD.get(codigo_ru, codigo_ru))


def generar_nombre_balance(codigo_ru: str, periodo: str) -> str:
    sociedad = obtener_nombre_sociedad(codigo_ru)
    periodo_nombre = convertir_periodo_a_nombre(periodo)

    partes = [sociedad, "Balance", periodo_nombre]
    if INCLUIR_TIMESTAMP:
        partes.append(TIMESTAMP_EJECUCION)

    return "_".join(partes) + ".txt"


def generar_nombre_saldos(codigo_fichero: str, periodo: str) -> str:
    periodo_nombre = convertir_periodo_a_nombre(periodo)

    partes = [limpiar_nombre(codigo_fichero), "Saldos_4_7", periodo_nombre]
    if INCLUIR_TIMESTAMP:
        partes.append(TIMESTAMP_EJECUCION)

    return "_".join(partes) + ".txt"


def _mover_part_file(ruta_temporal: str, ruta_final: str, extension: str) -> None:
    archivos_temporales = fs.ls(ruta_temporal)

    ficheros_part = [
        archivo for archivo in archivos_temporales
        if archivo.name.startswith("part-")
        and archivo.name.endswith(extension)
    ]

    if len(ficheros_part) != 1:
        nombres = [archivo.name for archivo in archivos_temporales]
        fs.rm(ruta_temporal, recurse=True)
        raise RuntimeError(
            "No se ha localizado exactamente un part-file. "
            f"Contenido encontrado: {nombres}"
        )

    if fs.exists(ruta_final):
        if SOBRESCRIBIR:
            fs.rm(ruta_final, recurse=False)
        else:
            fs.rm(ruta_temporal, recurse=True)
            raise FileExistsError(f"El fichero ya existe: {ruta_final}")

    resultado_movimiento = fs.mv(
        ficheros_part[0].path,
        ruta_final,
        create_path=True,
        overwrite=SOBRESCRIBIR,
    )

    if resultado_movimiento is False:
        fs.rm(ruta_temporal, recurse=True)
        raise RuntimeError(f"No se ha podido mover el fichero a {ruta_final}")

    fs.rm(ruta_temporal, recurse=True)

    if not fs.exists(ruta_final):
        raise RuntimeError(f"El fichero no existe después de moverlo: {ruta_final}")


def escribir_balance(dataframe: DataFrame, ruta_final: str) -> dict:
    """Escribe el fichero Magnitude separado por ; con cabecera."""
    identificador_temporal = uuid.uuid4().hex
    carpeta_destino = ruta_final.rsplit("/", 1)[0]
    ruta_temporal = f"{carpeta_destino}/_temp/{identificador_temporal}"

    fs.mkdirs(carpeta_destino)

    (
        dataframe
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .option("delimiter", ";")
        .option("encoding", "UTF-8")
        .option("nullValue", "")
        .option("emptyValue", "")
        .option("quote", "\u0000")
        .option("escape", "\u0000")
        .option("lineSep", "\n")
        .csv(ruta_temporal)
    )

    _mover_part_file(ruta_temporal, ruta_final, ".csv")

    return {
        "ruta": ruta_final,
        "filas": dataframe.count(),
        "tipo": "Balance",
    }


def formato_numero_es(valor) -> str:
    """1,234,567.89 -> 1.234.567,89"""
    numero = float(valor)
    texto = f"{abs(numero):,.2f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


def linea_saldos(periodo: str, cuenta: str, descripcion: str, saldo) -> str:
    descripcion = (descripcion or "")[:40]
    saldo = float(saldo or 0)

    debe = formato_numero_es(saldo) if saldo > 0 else ""
    haber = formato_numero_es(saldo) if saldo < 0 else ""

    if saldo > 0:
        saldo_txt = formato_numero_es(saldo)
    elif saldo < 0:
        saldo_txt = f"({formato_numero_es(saldo)})"
    else:
        saldo_txt = "0,00"

    # Anchuras tomadas del informe histórico.
    # Los positivos/cero ocupan una posición menos en la última columna
    # que los negativos entre paréntesis, igual que el reporte original.
    ancho_saldo = 17 if saldo < 0 else 16

    return (
        f" {periodo:<10}   "
        f"{cuenta:<10}   "
        f"{descripcion:<40} "
        f"{debe:>17} "
        f"{haber:>17}   "
        f"{saldo_txt:>{ancho_saldo}}"
    )


def fecha_cabecera_saldos() -> str:
    ahora = datetime.now()
    mes = MESES_ES[ahora.month]
    return f"{ahora.day:02d}-{mes}-{str(ahora.year)[-2:]} {ahora:%H:%M:%S}"


def construir_saldos_4_7(filas: list, sociedad: str, periodo_texto: str) -> list:
    """
    Construye las líneas del informe de ancho fijo.
    Reproduce columnas, signos, cabeceras y form feeds iniciales.
    No intenta replicar la paginación exacta del informe Oracle antiguo.
    """
    encabezado = (
        " Periodo      Cuenta       Descripción                                            "
        "Debe             Haber              Saldo"
    )
    separador = (
        " ----------   ----------   ----------------------------------------  "
        "----------------- -----------------  -----------------"
    )

    lineas = [
        f"{fecha_cabecera_saldos():>123}",
        "\f",
        f" {sociedad}",
        "\f",
        "",
        "",
        "",
        encabezado,
        separador,
    ]

    for fila in filas:
        lineas.append(
            linea_saldos(
                periodo=periodo_texto,
                cuenta=fila["Cuenta"],
                descripcion=fila["Descripcion"],
                saldo=fila["Saldo"],
            )
        )

    return lineas


def escribir_lineas_txt(lineas: list, ruta_final: str) -> dict:
    """
    Escribe un TXT de ancho fijo como un único part-file.
    Se añade CR a cada registro para producir CRLF al escribir con Spark.
    """
    identificador_temporal = uuid.uuid4().hex
    carpeta_destino = ruta_final.rsplit("/", 1)[0]
    ruta_temporal = f"{carpeta_destino}/_temp/{identificador_temporal}"

    fs.mkdirs(carpeta_destino)

    # Spark .text añade LF. Añadimos CR al final de cada línea para obtener CRLF.
    registros = [(linea + "\r",) for linea in lineas]
    df_lineas = spark.createDataFrame(registros, ["value"])

    (
        df_lineas
        .coalesce(1)
        .write
        .mode("overwrite")
        .text(ruta_temporal)
    )

    _mover_part_file(ruta_temporal, ruta_final, ".txt")

    return {
        "ruta": ruta_final,
        "filas": max(len(lineas) - 9, 0),
        "tipo": "Saldos_4_7",
    }


StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 10, Finished, Available, Finished, False)

In [9]:
# ============================================================
# GENERACIÓN DE LOS 10 TXT
# ============================================================

resultados = []

# ------------------------------------------------------------
# A. BALANCE MAGNITUDE: 1 fichero por D_RU
# ------------------------------------------------------------

sociedades_balance = (
    df.select("D_RU")
    .distinct()
    .orderBy("D_RU")
    .collect()
)

print(f"Balances a generar: {len(sociedades_balance)}")

for fila_sociedad in sociedades_balance:
    codigo_ru = fila_sociedad["D_RU"]

    df_sociedad = (
        df.filter(F.col("D_RU") == codigo_ru)
        .withColumn(
            "_OrdenCuenta",
            F.when(F.col("D_AC") == "P12000", F.lit(4))
             .when(F.col("D_AC").startswith("A"), F.lit(1))
             .when(F.col("D_AC").startswith("P"), F.lit(2))
             .when(F.col("D_AC").startswith("R"), F.lit(3))
             .otherwise(F.lit(9))
        )
        .orderBy("_OrdenCuenta", "D_AC")
        .select(*COLUMNAS_TXT)
    )

    periodos = [
        fila["D_DP"]
        for fila in df_sociedad.select("D_DP").distinct().collect()
    ]

    if len(periodos) != 1:
        raise ValueError(
            f"La sociedad {codigo_ru} contiene {len(periodos)} periodos: {periodos}"
        )

    periodo = periodos[0]
    nombre_fichero = generar_nombre_balance(codigo_ru, periodo)
    ruta_final = f"{CARPETA_SALIDA}/{periodo}/{nombre_fichero}"

    print(f"Generando Balance {codigo_ru}: {ruta_final}")

    resultado = escribir_balance(df_sociedad, ruta_final)
    resultado.update({
        "sociedad": codigo_ru,
        "periodo": periodo,
        "fichero": nombre_fichero,
    })
    resultados.append(resultado)


# ------------------------------------------------------------
# B. SALDOS 4_7: 1 fichero por CodigoFichero
# ------------------------------------------------------------

sociedades_saldos = (
    df_saldos.select("CodigoFichero")
    .distinct()
    .orderBy("CodigoFichero")
    .collect()
)

print(f"Saldos 4_7 a generar: {len(sociedades_saldos)}")

for fila_sociedad in sociedades_saldos:
    codigo_fichero = fila_sociedad["CodigoFichero"]

    df_sociedad = (
        df_saldos
        .filter(F.col("CodigoFichero") == codigo_fichero)
        .orderBy("Cuenta")
    )

    metadatos = (
        df_sociedad
        .select("Sociedad", "PeriodoTexto", "PeriodoCodigo")
        .distinct()
        .collect()
    )

    if len(metadatos) != 1:
        raise ValueError(
            f"{codigo_fichero} contiene metadatos inconsistentes: {metadatos}"
        )

    sociedad = metadatos[0]["Sociedad"]
    periodo_texto = metadatos[0]["PeriodoTexto"]
    periodo = metadatos[0]["PeriodoCodigo"]

    filas = (
        df_sociedad
        .select("Cuenta", "Descripcion", "Saldo")
        .orderBy("Cuenta")
        .collect()
    )

    lineas = construir_saldos_4_7(
        filas=filas,
        sociedad=sociedad,
        periodo_texto=periodo_texto,
    )

    nombre_fichero = generar_nombre_saldos(codigo_fichero, periodo)
    ruta_final = f"{CARPETA_SALIDA}/{periodo}/{nombre_fichero}"

    print(f"Generando Saldos 4_7 {codigo_fichero}: {ruta_final}")

    resultado = escribir_lineas_txt(lineas, ruta_final)
    resultado.update({
        "sociedad": codigo_fichero,
        "periodo": periodo,
        "fichero": nombre_fichero,
    })
    resultados.append(resultado)


# ------------------------------------------------------------
# Control esperado: 5 Balance + 5 Saldos_4_7
# ------------------------------------------------------------

print(f"Total de ficheros generados: {len(resultados)}")

if len(sociedades_balance) == 5 and len(sociedades_saldos) == 5:
    if len(resultados) != 10:
        raise RuntimeError(
            f"Se esperaban 10 ficheros y se han generado {len(resultados)}."
        )


StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 11, Finished, Available, Finished, False)

Balances a generar: 5
Generando Balance ESP066: Files/Exportaciones/Balance/2026.03/TINKLE_COMM_Balance_MAR-26_20260821_023752.txt
Generando Balance ESP067: Files/Exportaciones/Balance/2026.03/TINKLE_EU_Balance_MAR-26_20260821_023752.txt
Generando Balance ESP068: Files/Exportaciones/Balance/2026.03/TINKLE_JAZZ_Balance_MAR-26_20260821_023752.txt
Generando Balance ESP069: Files/Exportaciones/Balance/2026.03/TINKLE_ONION_Balance_MAR-26_20260821_023752.txt
Generando Balance ESP070: Files/Exportaciones/Balance/2026.03/TINKLE_CONS_Balance_MAR-26_20260821_023752.txt
Saldos 4_7 a generar: 5
Generando Saldos 4_7 TINKLE_COMM: Files/Exportaciones/Balance/2026.03/TINKLE_COMM_Saldos_4_7_MAR-26_20260821_023752.txt
Generando Saldos 4_7 TINKLE_CONS: Files/Exportaciones/Balance/2026.03/TINKLE_CONS_Saldos_4_7_MAR-26_20260821_023752.txt
Generando Saldos 4_7 TINKLE_EU: Files/Exportaciones/Balance/2026.03/TINKLE_EU_Saldos_4_7_MAR-26_20260821_023752.txt
Generando Saldos 4_7 TINKLE_JAZZ: Files/Exportaciones/

In [10]:
# ============================================================
# RESUMEN DE EJECUCIÓN
# ============================================================

print("")
print("=" * 100)
print("FICHEROS GENERADOS")
print("=" * 100)

for resultado in resultados:
    print(
        f"Tipo: {resultado['tipo']} | "
        f"Sociedad: {resultado['sociedad']} | "
        f"Periodo: {resultado['periodo']} | "
        f"Filas: {resultado['filas']:,}"
    )
    print(f"Ruta: {resultado['ruta']}")
    print("-" * 100)

print(f"TOTAL: {len(resultados)} fichero(s)")

# Vista previa de un Balance y un Saldos_4_7.
for tipo in ["Balance", "Saldos_4_7"]:
    candidatos = [r for r in resultados if r["tipo"] == tipo]
    if candidatos:
        ruta = candidatos[0]["ruta"]
        print("")
        print(f"Vista previa {tipo}: {ruta}")
        print("=" * 100)
        print(fs.head(ruta, 4000))


StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 12, Finished, Available, Finished, False)


FICHEROS GENERADOS
Tipo: Balance | Sociedad: ESP066 | Periodo: 2026.03 | Filas: 60
Ruta: Files/Exportaciones/Balance/2026.03/TINKLE_COMM_Balance_MAR-26_20260821_023752.txt
----------------------------------------------------------------------------------------------------
Tipo: Balance | Sociedad: ESP067 | Periodo: 2026.03 | Filas: 34
Ruta: Files/Exportaciones/Balance/2026.03/TINKLE_EU_Balance_MAR-26_20260821_023752.txt
----------------------------------------------------------------------------------------------------
Tipo: Balance | Sociedad: ESP068 | Periodo: 2026.03 | Filas: 38
Ruta: Files/Exportaciones/Balance/2026.03/TINKLE_JAZZ_Balance_MAR-26_20260821_023752.txt
----------------------------------------------------------------------------------------------------
Tipo: Balance | Sociedad: ESP069 | Periodo: 2026.03 | Filas: 36
Ruta: Files/Exportaciones/Balance/2026.03/TINKLE_ONION_Balance_MAR-26_20260821_023752.txt
------------------------------------------------------------------

In [11]:
# ============================================================
# RESULTADO PARA EL PIPELINE
# ============================================================

import json

resultado_pipeline = {
    "estado": "OK",
    "numero_ficheros": len(resultados),
    "numero_balance": len([r for r in resultados if r["tipo"] == "Balance"]),
    "numero_saldos_4_7": len([r for r in resultados if r["tipo"] == "Saldos_4_7"]),
    "ficheros": resultados,
    "fecha_ejecucion": datetime.now().isoformat(),
}

resultado_json = json.dumps(resultado_pipeline, ensure_ascii=False)
print(resultado_json)

try:
    notebookutils.notebook.exit(resultado_json)
except NameError:
    try:
        mssparkutils.notebook.exit(resultado_json)
    except Exception:
        pass


StatementMeta(, 764b06b5-09f8-4282-9f78-473785ed1cc1, 13, Finished, Available, Finished, False)

{"estado": "OK", "numero_ficheros": 10, "numero_balance": 5, "numero_saldos_4_7": 5, "ficheros": [{"ruta": "Files/Exportaciones/Balance/2026.03/TINKLE_COMM_Balance_MAR-26_20260821_023752.txt", "filas": 60, "tipo": "Balance", "sociedad": "ESP066", "periodo": "2026.03", "fichero": "TINKLE_COMM_Balance_MAR-26_20260821_023752.txt"}, {"ruta": "Files/Exportaciones/Balance/2026.03/TINKLE_EU_Balance_MAR-26_20260821_023752.txt", "filas": 34, "tipo": "Balance", "sociedad": "ESP067", "periodo": "2026.03", "fichero": "TINKLE_EU_Balance_MAR-26_20260821_023752.txt"}, {"ruta": "Files/Exportaciones/Balance/2026.03/TINKLE_JAZZ_Balance_MAR-26_20260821_023752.txt", "filas": 38, "tipo": "Balance", "sociedad": "ESP068", "periodo": "2026.03", "fichero": "TINKLE_JAZZ_Balance_MAR-26_20260821_023752.txt"}, {"ruta": "Files/Exportaciones/Balance/2026.03/TINKLE_ONION_Balance_MAR-26_20260821_023752.txt", "filas": 36, "tipo": "Balance", "sociedad": "ESP069", "periodo": "2026.03", "fichero": "TINKLE_ONION_Balance_MA